# AWS S3 → SNS → SQS → Snowpipe: Baby Steps Checklist

---

## AWS (UAT Account) Side

### 1. Create SNS Topic

```
aws sns create-topic --name s3-file-events-topic
```

* *Save the Topic ARN from the output.*

---

### 2. Create SQS Queue (Standard)

```
aws sqs create-queue --queue-name snowpipe-file-events-queue
```

* *Save the Queue URL and ARN from the output.*

---

### 3. Subscribe SQS Queue to SNS Topic

```
aws sns subscribe \
  --topic-arn arn:aws:sns:your-sns-topic-arn \
  --protocol sqs \
  --notification-endpoint arn:aws:sqs:your-region:your-account-id:snowpipe-file-events-queue
```

---

### 4. Update SQS Queue Policy (Add Permissions)

* **a.** Allow SNS to send messages to SQS:

```json
{
  "Effect": "Allow",
  "Principal": "*",
  "Action": "sqs:SendMessage",
  "Resource": "arn:aws:sqs:your-region:your-account-id:snowpipe-file-events-queue",
  "Condition": {
    "ArnEquals": {
      "aws:SourceArn": "arn:aws:sns:your-region:your-account-id:s3-file-events-topic"
    }
  }
}
```

* **b.** Allow Snowflake’s AWS account to poll and delete messages:

```json
{
  "Effect": "Allow",
  "Principal": { "AWS": "arn:aws:iam::<SNOWFLAKE_AWS_ACCOUNT_ID>:root" },
  "Action": [
    "sqs:ReceiveMessage",
    "sqs:DeleteMessage",
    "sqs:GetQueueAttributes"
  ],
  "Resource": "arn:aws:sqs:your-region:your-account-id:snowpipe-file-events-queue"
}
```

* *Combine both into your SQS policy (via Console or CLI).*

---

### 5. Configure S3 Event Notifications to SNS Topic

* **Console:**

  * S3 bucket → Properties → Event notifications
  * New event: All object create events
  * Destination: SNS topic
* **CLI:**

```
aws s3api put-bucket-notification-configuration \
  --bucket your-bucket-name \
  --notification-configuration '{
    "TopicConfigurations": [{
      "TopicArn": "arn:aws:sns:your-region:your-account-id:s3-file-events-topic",
      "Events": ["s3:ObjectCreated:*"]
    }]
}'
```

---

### 6. Share SQS Queue ARN with Snowflake Team

* *They need this for Snowpipe setup.*

---

## Snowflake Side

### 7. Create/Config PIPE with AUTO\_INGEST

```sql
CREATE OR REPLACE PIPE my_db.my_schema.my_pipe
  AUTO_INGEST = TRUE
  AS COPY INTO my_table
  FROM @my_stage
  FILE_FORMAT = (TYPE = 'PARQUET');
```

* *Specify your SQS queue ARN as the notification channel.*

---

## Validation Steps

* Drop a test file into S3.
* Confirm event appears in SQS (check SQS metrics).
* Snowflake should auto-ingest and log the load.

---

## Summary Table

| Step | Party     | Action                          | Notes                        |
| ---- | --------- | ------------------------------- | ---------------------------- |
| 1    | AWS (UAT) | Create SNS Topic                | Save ARN                     |
| 2    | AWS (UAT) | Create SQS Queue                | Save ARN & URL               |
| 3    | AWS (UAT) | Subscribe SQS to SNS            |                              |
| 4    | AWS (UAT) | Set SQS Policy                  | SNS publish & Snowflake poll |
| 5    | AWS (UAT) | Configure S3 Event → SNS        | All object create events     |
| 6    | AWS (UAT) | Share SQS ARN with Snowflake    |                              |
| 7    | Snowflake | Create/Config PIPE AUTO\_INGEST | Use provided SQS ARN         |
| 8    | Both      | Test end-to-end ingestion       | S3 → SQS → Snowflake PIPE    |
